In [2]:
from google.colab import drive, userdata
import os

In [4]:
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive"
PROJECT = f"{DRIVE}/abaya_bot_project"
os.makedirs(PROJECT, exist_ok=True)

Mounted at /content/drive


In [5]:
os.environ["GEMINI_API_KEY"] = userdata.get("bot2")

In [6]:
!pip install -q openai sentence-transformers pillow numpy google-genai

In [7]:
for f in ("catalog.json", "index.npy", "store_info.md", "images"):
    p = f"{PROJECT}/{f}"
    print(f, "←", " ✅" if os.path.exists(p) else " ❌")

catalog.json ←  ✅
index.npy ←  ✅
store_info.md ←  ✅
images ←  ✅


In [23]:
%%writefile /content/drive/MyDrive/abaya_bot_project/bot_core.py
import os, base64, io, json, time
import numpy as np
from PIL import Image
from openai import OpenAI, APIStatusError, APIConnectionError, APITimeoutError
from sentence_transformers import SentenceTransformer
from google.colab import userdata

PROJECT = os.path.dirname(os.path.abspath(__file__))

# if the first besy try the others
MODELS = ["gemini-3.8-flash", "gemini-3.5-flash", "gemini-3.5-flash-lite", "gemini-3.1-flash-lite", "gemini-3.1-pro-preview"]
TOP_K = 3

SYSTEM = """أنتِ مساعدة مبيعات لمحل عبايات، تردين على الزبونات بلهجة خليجية بسيطة ولبقة ومختصرة.
معلومات المحل:
{store_info}
الكاتلوج (مصدرك الوحيد للأسعار والتوفر):
{catalog_text}
طريقة الشغل:
- ردي مباشرة على السلام والشكر والأسئلة عن الأسعار والألوان والمقاسات والتوصيل والاستبدال. لا تحوّلين هذي للموظفة أبداً.
- إذا الزبونة طرشت صورة، بتوصلك صورتها وبعدها صور أقرب عبايات من الكاتلوج مع كودها ودرجة التشابه.
  قارني بدقة: التطريز، الأكمام، القصّة، الأزرار، القماش، اللون. درجة التشابه مجرد مساعدة، قرارك على المقارنة البصرية.
  * إذا متأكدة: جاوبي باسم العباية وسعرها وألوانها ومقاساتها.
  * إذا عبايتين قريبتين وما تقدرين تفرقين: اعرضي الاثنتين واسألي الزبونة أي وحدة تقصد.
  * إذا ولا وحدة تطابق: قولي إنها ما لقيتها بالكاتلوج، واسألي إذا تبي الموظفة تتأكد لها.
    وإذا وافقت، استخدمي handoff_to_staff.
- لا تخمنين سعر أو معلومة مو موجودة فوق.
- handoff_to_staff فقط: شكوى، تفصيل خاص، مشكلة في طلب أو دفع، أو سؤال ما له جواب بالمعلومات.
- الرد قصير (2 إلى 4 أسطر).
"""

TOOLS = [{
    "type": "function",
    "function": {
        "name": "handoff_to_staff",
        "description": "حوّل المحادثة للموظفة. فقط للشكاوى، التفصيل الخاص، مشاكل الطلبات/الدفع، أو أسئلة ما لها جواب في المعلومات المعطاة.",
        "parameters": {
            "type": "object",
            "properties": {"reason": {"type": "string", "description": "سبب التحويل بجملة قصيرة"}},
            "required": ["reason"],
        },
    },
}]

RETRY_CODES = (429, 500, 502, 503, 504)
SKIP_MODEL_CODES = (400, 404)

def get_api_key():
    key = os.environ.get("GEMINI_API_KEY")
    if key:
        return key
    try:
        from google.colab import userdata
        return userdata.get("bot2")
    except Exception:
        raise RuntimeError("Place the key GEMINI_API_KEY")


def to_block(img: Image.Image) -> dict:
    img = img.convert("RGB")
    img.thumbnail((1024, 1024))
    buf = io.BytesIO()
    img.save(buf, "JPEG", quality=85)
    b64 = base64.b64encode(buf.getvalue()).decode()
    return {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}"}}

# the index and cataloge load once
class _Shared:
    loaded = False

    @classmethod
    def load(cls):
        if cls.loaded:
            return
        cls.catalog = json.load(open(f"{PROJECT}/catalog.json", encoding="utf-8-sig"))
        store_info = open(f"{PROJECT}/store_info.md", encoding="utf-8-sig").read()
        cls.emb = np.load(f"{PROJECT}/index.npy")
        assert len(cls.emb) == len(cls.catalog), "Embedding size mismatch with catalog!"
        cls.clip = SentenceTransformer("clip-ViT-B-32")

        catalog_text = "\n".join(
            f"- {c['code']} | {c['name']} | {c['price']} | ألوان: {c['colors']} | "
            f"مقاسات: {c['sizes']} | قماش: {c.get('fabric', '')} | "
            f"نوع الطلب: {c.get('order_type', '')} | مدة التفصيل: {c.get('tailoring_days', '')} | "
            f"التوفر: {c.get('stock', '')} | {c['description']}"
            for c in cls.catalog)

        cls.system = SYSTEM.format(store_info=store_info, catalog_text=catalog_text)
        cls.client = OpenAI(
            api_key=get_api_key(),
            base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
            max_retries=0,
            timeout=60, )
        cls.loaded = True

# for each customer alone
class AbayaBot:
    def __init__(self):
        _Shared.load()
        self.history = []

    def find_similar(self, img, k=TOP_K):
        q = _Shared.clip.encode(img.convert("RGB"), normalize_embeddings=True)
        sims = _Shared.emb @ q
        return [(_Shared.catalog[i], float(sims[i])) for i in np.argsort(-sims)[:k]]
# try each model 3 times
    def _call(self, messages):
        last_error = None
        for model in MODELS:
            for attempt in range(3):
                try:
                    return _Shared.client.chat.completions.create(
                        model=model, max_tokens=800, tools=TOOLS, messages=messages)
                except APIStatusError as e:
                    last_error = e
                    if e.status_code in SKIP_MODEL_CODES:
                        break                      # ينتقل للموديل اللي بعده مباشرة
                    if e.status_code not in RETRY_CODES:
                        raise
                except (APIConnectionError, APITimeoutError) as e:
                    last_error = e
                time.sleep(2 * 2 ** attempt)
            print(f"⚠️ {model} didn't work ttry the others")
        raise last_error

    def reply(self, text="", image_bytes=None):
        content = []
        if image_bytes:
            img = Image.open(io.BytesIO(image_bytes))
            content += [{"type": "text", "text": " customer photos:"}, to_block(img)]
            for c, score in self.find_similar(img):
                with Image.open(f"{PROJECT}/images/{c['image']}") as cand:
                    content += [
                        {"type": "text",
                         "text": f" Recommended from the catalog : {c['code']} ({c['name']}) - similar to  {score:.2f}"},
                        to_block(cand),
                    ]
        content.append({"type": "text", "text": text or "customer sent photo without text"})

        start = len(self.history)
        self.history.append({"role": "user", "content": content})
        handoff = None

        try:
            while True:
                resp = self._call([{"role": "system", "content": _Shared.system}] + self.history)
                msg = resp.choices[0].message
                self.history.append(msg.model_dump(exclude_none=True))
                if not msg.tool_calls:
                    break
                for call in msg.tool_calls:
                    if call.function.name == "handoff_to_staff":
                        try:
                            handoff = json.loads(call.function.arguments).get("reason", "")
                        except Exception:
                            handoff = "غير مححدد"
                    self.history.append({"role": "tool", "tool_call_id": call.id,
                                         "content": "تم إشعار الموظفة."})
        except Exception:
            del self.history[start:]

        self.history[start]["content"] = [b for b in self.history[start]["content"] if b["type"] == "text"]
        return msg.content or "", handoff

Overwriting /content/drive/MyDrive/abaya_bot_project/bot_core.py


notifications

In [19]:
%%writefile /content/drive/MyDrive/abaya_bot_project/notifications.py
import os, smtplib
from email.message import EmailMessage

STAFF_EMAIL = os.environ.get("STAFF_EMAIL", "")
SENDER = os.environ.get("SENDER_EMAIL", "")
SENDER_PASS = os.environ.get("GMAIL_APP_PASSWORD", "")


def notify_staff(customer_id, reason, last_message):
    """يطرش إيميل للموظفة. إذا فشل ما يوقف البوت."""
    if not (STAFF_EMAIL and SENDER and SENDER_PASS):
        print("⚠️ إعدادات الإيميل ناقصة، ما انرسل إشعار")
        return False
    try:
        msg = EmailMessage()
        msg["Subject"] = f"طلب تحويل من زبونة: {customer_id}"
        msg["From"] = SENDER
        msg["To"] = STAFF_EMAIL
        msg.set_content(
            f"رقم الزبونة: {customer_id}\n"
            f"السبب: {reason}\n"
            f"آخر رسالة: {last_message}\n"
        )
        with smtplib.SMTP_SSL("smtp.gmail.com", 465, timeout=20) as s:
            s.login(SENDER, SENDER_PASS)
            s.send_message(msg)
        return True
    except Exception as e:
        print("⚠️ فشل إرسال الإيميل:", e)
        return False

Overwriting /content/drive/MyDrive/abaya_bot_project/notifications.py


In [20]:
os.environ["STAFF_EMAIL"] = "noorbintahmd@gmail.com"
os.environ["SENDER_EMAIL"] = "noorbintahmd@gmail.com"
os.environ["GMAIL_APP_PASSWORD"] = userdata.get("gmail_pass")

print("Email settings loaded!")

Email settings loaded!


In [21]:
import sys, importlib
sys.path.insert(0, PROJECT)
import notifications
importlib.reload(notifications)

ok = notifications.notify_staff("تجربة-Colab", "اختبار الإشعار", "هذي رسالة تجريبية")
print(" ✅" if ok else " ❌")

 ✅


## تجربة البوت

In [24]:
import sys, importlib
sys.path.insert(0, PROJECT)
import bot_core, notifications
importlib.reload(bot_core)
importlib.reload(notifications)
from bot_core import AbayaBot

bot = AbayaBot()
print(bot.reply("السلام عليكم")[0])
print("-----")
print(bot.reply("شنو ألوان وأسعار العبايات عندكم؟")[0])

modules.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/1.91k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.03k [00:00<?, ?B/s]

0_CLIPModel/model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

0_CLIPModel/model.safetensors: downloading bytes:           |  0.00B            

... (3 outputs left out)

vocab.json:   0%|          | 0.00/961k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

⚠️ gemini-3.8-flash didn't work ttry the others
وعليكم السلام والرحمة! 👋 
أهلاً بيج في متجر ايرل 🌸 
آمريني أختي، شلون أقدر أساعدج اليوم؟
-----
⚠️ gemini-3.8-flash didn't work ttry the others
أسعار عباياتنا تترواح بين 15 إلى 27 دينار بحريني حسب الموديل والقماش ✨ 

وعندنا تشكيلة ألوان متنوعة مثل الأسود، البني، الكحلي، الماروني، البنفسجي وغيرها 🎨

إذا في خاطرج موديل معين (كلوش، ستريت، بشت، بليزر) أو حابة تسألين عن كود عباية محدد، اعطيني خبر وبعطيج كل التفاصيل 💕


Transfer test for the employee

In [25]:
bot2 = AbayaBot()
for q in ["طلبي وصل غلط وابي أكلم مسؤول", "ابي اغير القماش"]:
    answer, handoff = bot2.reply(q)
    print(f"س: {q}\nج: {answer}\nتحويل: {handoff}\n")
    if handoff:
        notifications.notify_staff("تجربة-Colab", handoff, q)

⚠️ gemini-3.8-flash didn't work ttry the others
⚠️ gemini-3.8-flash didn't work ttry the others
س: طلبي وصل غلط وابي أكلم مسؤول
ج: يا هلا بوج حبيبتي، مسامحة على اللي صار 🙏

بخلي الموظفة تتأكد لج من هالتفاصيل، وبترد عليج قريب إن شاء الله.
تحويل: استلام طلب غلط وطلب التحدث مع المسؤول

⚠️ gemini-3.8-flash didn't work ttry the others
⚠️ gemini-3.8-flash didn't work ttry the others
س: ابي اغير القماش
ج: إن شاء الله حبيبتي ✨

بخلي الموظفة تتأكد لج من هالتفاصيل، وبترد عليج قريب إن شاء الله.
تحويل: طلب تغيير القماش على العباية



test with photo

In [26]:
from google.colab import files

bot3 = AbayaBot()
uploaded = files.upload()
img_bytes = uploaded[list(uploaded.keys())[0]]
answer, handoff = bot3.reply("بكم هذي العباية؟", img_bytes)
print(answer, "\nتحويل:", handoff)

Saving 892255094_18137047081788007_6698510324677332243_n..jpg to 892255094_18137047081788007_6698510324677332243_n..jpg
⚠️ gemini-3.8-flash didn't work ttry the others
 عباية بقصة نص بشت قلبتين).
- Price: 20 دينار بحريني.
- Available colors: بني، بنفس 
تحويل: None
